ARTI308 - Machine Learning

# Credit Card Customer Segmentation Project

In this project, you will use K-Means clustering to segment [credit card customers](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata/data) based on their usage behavior. This is an unsupervised learning problem because the dataset does not contain a target label for customer groups.

You will use the `CC_GENERAL.csv` dataset.

## About the Dataset

The dataset contains customer-level credit card usage behavior. Each row represents one credit card holder, and the columns describe different behavioral variables such as balance, purchases, cash advance, payments, and tenure. The goal is to group similar customers together so that the company can understand different customer segments and design better marketing strategies.

## Import Libraries

**Import the libraries you need for data analysis, visualization, preprocessing, clustering, and evaluation.**

In [ ]:
 import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
matplotlib inline

## Get the Data

**Read the `CC_GENERAL.csv` file and save it in a dataframe called `df`.**

In [ ]:
 df = pd.read_csv('/kaggle/input/datasets/kadisu/cc-gen/CC_GENERAL.csv')

**Check the first five rows of the dataset.**

In [ ]:
 df.head()

**Check the shape of the dataset.**

In [ ]:
 df.shape

**Check basic information about the dataset using `info()`.**

In [ ]:
df.info() 

**Check summary statistics using `describe()`.**

In [ ]:
 df.describe()

## Data Cleaning

The column `CUST_ID` is an identification column. It is not useful for clustering because it does not describe customer behavior.

**Drop the `CUST_ID` column from the dataframe.**

In [ ]:
df = df.drop(columns=['CUST_ID'])

**Check the missing values in each column.**

In [ ]:
 df.isnull().sum()

In [ ]:
df.fillna(df.mean(), inplace=True)

Some columns may contain missing values.

Hint: You can handle missing values by either:
- filling them with the mean value
- or dropping the rows that contain missing values

For this project, use mean imputation.

**Fill the missing values with the mean of each column.**

In [ ]:
df.fillna(df.mean(), inplace=True)

**Check the missing values again to make sure they were handled.**

In [ ]:
df.isnull().sum()

## Exploratory Data Analysis

Before applying clustering, it is important to understand the data.

**Create histograms for the numerical columns.**

In [ ]:
 df.hist(figsize=(15, 12), bins=20)
plt.tight_layout()
plt.show()

**Create a correlation heatmap to understand relationships between the features.**

In [ ]:
 plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.tight_layout()
plt.show()

**Create a scatter plot between `BALANCE` and `PURCHASES`.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['BALANCE'], df['PURCHASES'], alpha=0.5, c='blue', edgecolors='w')
plt.xlabel('BALANCE')
plt.ylabel('PURCHASES')
plt.title('Scatter Plot: BALANCE vs PURCHASES')
plt.show()

**Create a scatter plot between `BALANCE` and `CASH_ADVANCE`.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['BALANCE'], df['CASH_ADVANCE'], alpha=0.5, c='red', edgecolors='w')
plt.xlabel('BALANCE')
plt.ylabel('CASH_ADVANCE')
plt.title('Scatter Plot: BALANCE vs CASH_ADVANCE')
plt.show()

## Feature Scaling

K-Means is a distance-based algorithm. Therefore, feature scaling is very important.

The features in this dataset have very different ranges. For example, `BALANCE`, `PURCHASES`, and `CREDIT_LIMIT` may have large values, while frequency columns are between 0 and 1.

**Use StandardScaler to scale the data. Save the scaled data in a variable called `X_scaled`.**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

## Choosing K Intuitively

Choosing K is one of the most difficult parts of K-Means.

Since this dataset has many features, it is not easy to visually see the clusters directly.

However, we can still compare different K values using the elbow method and silhouette score.

## Elbow Method

**Create a loop that fits K-Means models for K values from 1 to 10. Save the inertia values in a list called `inertia_values`.**

In [ ]:
inertia_values = []
K_range = range(1, 11)

for k in K_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertia_values.append(model.inertia_) 

**Plot the elbow curve.**

In [ ]:
 plt.figure(figsize=(8, 5))
plt.plot(K_range, inertia_values, marker='o', linestyle='-', color='b')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method For Optimal K')
plt.xticks(K_range)
plt.grid(True)
plt.show()

**Output Interpretation**

Look at the elbow curve and try to identify where the decrease in inertia starts to slow down.

That point can suggest a reasonable value for K.

## Silhouette Score

The silhouette score helps evaluate how well-separated the clusters are.

**Create a loop that calculates the silhouette score for K values from 2 to 10. Save the scores in a list called `silhouette_scores`.**

In [ ]:
silhouette_scores = []
K_range_sil = range(2, 11)

for k in K_range_sil:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

**Plot the silhouette scores.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(K_range_sil, silhouette_scores, marker='o', linestyle='-', color='g')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Scores For Different K Values')
plt.xticks(K_range_sil)
plt.grid(True)
plt.show()

**Create a table showing each K value and its silhouette score.**

In [ ]:
score_table = pd.DataFrame({
    'K': list(K_range_sil),
    'Silhouette Score': silhouette_scores
})
score_table

**Output Interpretation**

A higher silhouette score usually means better clustering.

However, do not rely only on the highest value. Also consider whether the chosen K makes sense for customer segmentation.

## Create the Final K-Means Model

**Based on the elbow curve and silhouette scores, choose a final K value. Then train a final K-Means model.**

Use `random_state=42` and `n_init=10`.

In [ ]:
final_k = 4
final_model = KMeans(n_clusters=final_k, random_state=42, n_init=10)
final_model.fit(X_scaled)

**Add the final cluster labels to the original dataframe in a new column called `Cluster`.**

In [ ]:
df['Cluster'] = final_model.labels_

**Check the first five rows after adding the cluster labels.**

In [ ]:
df.head()

## Cluster Analysis

Now we need to understand what each cluster means.

**Create a summary table using `groupby()` to show the mean values of each feature for each cluster.**

In [ ]:
cluster_summary = df.groupby('Cluster').mean()
cluster_summary

**Check how many customers are in each cluster.**

In [ ]:
 df['Cluster'].value_counts().sort_index()

## Visualizing the Final Clusters

Since the dataset has many features, we will use PCA to reduce the data into two components only for visualization.

This visualization does not replace the original clustering. It only helps us see the clusters in a 2D plot.

**Use PCA with 2 components and plot the clusters.**

In [ ]:
 pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df['Cluster'], palette='viridis', alpha=0.6)
plt.title('Visualizing Customer Segments using PCA')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Cluster')
plt.show()

**Output Interpretation**

The PCA plot gives a simplified 2D view of the clusters.

If the clusters are not perfectly separated, that is normal because the original dataset has many features and the plot only shows two compressed dimensions.

## Final Questions

Answer the following questions:

1. Why is this an unsupervised learning problem?

It is an unsupervised learning problem because the dataset does not contain predefined labels or target classes for customers.

2. Why did we remove the `CUST_ID` column?

CUST_ID is only a unique identifier for each customer and does not provide meaningful behavioral information for clustering. Keeping it could introduce noise and negatively affect distance calculations used by K-Means.

3. Which columns had missing values?

CREDIT_LIMIT → 1 missing value
MINIMUM_PAYMENTS → 313 missing values

4. How did you handle the missing values?

The missing values were handled using mean imputation. Each missing value was replaced with the average value of its corresponding column using:
df.fillna(df.mean(), inplace=True)


5. Why is scaling important before applying K-Means?

Scaling is important because K-Means relies on Euclidean distance to form clusters. Features with larger numeric ranges, such as BALANCE or CREDIT_LIMIT, could dominate the clustering process and bias the results. Standardization ensures that all features contribute equally.


6. Which K value did you choose? Explain your answer using the elbow method and silhouette score.

We selected K = 4.

The Elbow Method showed a noticeable bend around K = 3 to K = 4, indicating that adding more clusters after this point provides only small improvements in inertia.

The Silhouette Score reached its highest value at K = 3 (0.2506), while K = 4 produced a slightly lower but still acceptable score (0.1976). We chose K = 4 because it provided more meaningful customer segmentation from a business perspective.

7. Based on the cluster summary table, describe each customer segment in your own words.

Cluster 0 — Active Value Buyers

These customers maintain relatively low balances but purchase frequently, especially through installment payments. They rarely rely on cash advances and actively use their credit cards for regular purchases.

Cluster 1 — High-Value Big Spenders

These are premium customers with very high balances and credit limits. They spend heavily and consistently while maintaining strong repayment behavior.

Cluster 2 — Cash Advance Reliant Customers

These customers rely heavily on cash advances rather than regular purchases. They usually maintain high balances and frequently withdraw cash using their credit cards.

Cluster 3 — Low-Activity / Dormant Customers

These customers have low balances, small credit limits, and minimal purchasing activity. Their overall engagement with the credit card is very limited.


8. Which cluster may represent high-value customers?

Cluster 1 represents high-value customers because they have the highest average credit limits, balances, and purchase amounts while maintaining good repayment behavior.


9. Which cluster may represent customers who rely more on cash advance?

Cluster 2 represents customers who rely heavily on cash advances because they show the highest average cash advance usage and withdrawal frequency.

10. How can a company use these clusters for marketing strategy?

Cluster 0 — Active Buyers

Offer cashback programs, loyalty rewards, and merchant discounts to encourage larger and more frequent purchases.

Cluster 1 — Big Spenders

Provide premium benefits such as VIP memberships, exclusive rewards, higher credit limits, and luxury services to improve retention.

Cluster 2 — Cash Advance Users

Promote lower-interest loan options or balance transfer offers to reduce reliance on expensive cash advances.

Cluster 3 — Dormant Customers

Use reactivation campaigns such as fee waivers, bonus points, or introductory offers to encourage card usage and increase engagement.